# Water Quality Testing Data Analysis

This notebook analyzes a dataset of 500 water samples across five quality parameters (pH, temperature, turbidity, dissolved oxygen, conductivity). The analysis includes exploratory data analysis, correlation studies, visualization, and predictive modeling.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import linear_model
import statsmodels.api as sm

sns.set(style="whitegrid")
%matplotlib inline

## 2. Data Loading and Inspection

In [ ]:
df = pd.read_csv("../data/water_quality_testing.csv")
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

## 3. Distribution Analysis

Examine the distribution of each water quality parameter using histograms with kernel density estimation (KDE).

In [ ]:
parameters = ["pH", "Temperature (°C)", "Turbidity (NTU)", "Dissolved Oxygen (mg/L)", "Conductivity (µS/cm)"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, param in enumerate(parameters):
    sns.histplot(df[param], kde=True, bins=30, ax=axes[i])
    axes[i].set_title(f"Distribution of {param}")
    axes[i].set_xlabel(param)
    axes[i].set_ylabel("Frequency")

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
numeric_cols = df.drop(columns=["Sample ID"])
correlation_matrix = numeric_cols.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".3f", linewidths=0.5)
plt.title("Correlation Matrix of Water Quality Parameters")
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(numeric_cols, kind="scatter", height=2.5)
plt.suptitle("Pair Plot of Water Quality Parameters", y=1.02)
plt.show()

## 5. pH vs Dissolved Oxygen

The strongest correlation in the dataset is between pH and dissolved oxygen (r = 0.705), suggesting a strong positive relationship.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df["pH"], df["Dissolved Oxygen (mg/L)"], color="green", alpha=0.5)

# Trend line
z = np.polyfit(df["pH"], df["Dissolved Oxygen (mg/L)"], 1)
p = np.poly1d(z)
plt.plot(df["pH"], p(df["pH"]), "r--", label="Trend line")

plt.title("pH vs Dissolved Oxygen with Trend Line")
plt.xlabel("pH")
plt.ylabel("Dissolved Oxygen (mg/L)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
correlation = df["pH"].corr(df["Dissolved Oxygen (mg/L)"])
print(f"Correlation coefficient between pH and Dissolved Oxygen: {correlation:.4f}")

## 6. Parameter Relationships

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.regplot(data=df, x="Turbidity (NTU)", y="Dissolved Oxygen (mg/L)", ax=axes[0, 0])
axes[0, 0].set_title("Turbidity vs Dissolved Oxygen")

sns.regplot(data=df, x="Temperature (°C)", y="Conductivity (µS/cm)", ax=axes[0, 1],
            line_kws={"color": "black"})
axes[0, 1].set_title("Temperature vs Conductivity")

sns.regplot(data=df, x="pH", y="Turbidity (NTU)", ax=axes[1, 0],
            scatter_kws={"color": "grey"}, line_kws={"color": "black"})
axes[1, 0].set_title("pH vs Turbidity")

sns.regplot(data=df, x="Dissolved Oxygen (mg/L)", y="Conductivity (µS/cm)", ax=axes[1, 1])
axes[1, 1].set_title("Dissolved Oxygen vs Conductivity")

plt.tight_layout()
plt.show()

## 7. Predictive Modeling: Conductivity

Build linear regression models to predict conductivity from other water quality parameters.

### Two-Feature Model (Turbidity + Dissolved Oxygen)

In [ ]:
features_2 = ["Turbidity (NTU)", "Dissolved Oxygen (mg/L)"]
X_2 = df[features_2]
y = df["Conductivity (µS/cm)"]

model_2 = linear_model.LinearRegression()
model_2.fit(X_2, y)

print(f"Intercept: {model_2.intercept_:.4f}")
print(f"Coefficients: {dict(zip(features_2, model_2.coef_))}")

In [ ]:
predictions_2 = model_2.predict(X_2)

results_2 = df[["Sample ID", "Turbidity (NTU)", "Dissolved Oxygen (mg/L)"]].copy()
results_2["Predicted Conductivity (µS/cm)"] = predictions_2
results_2.head(10)

In [ ]:
results_2_plot = results_2.copy()
results_2_plot["Conductivity (µS/cm)"] = predictions_2

sns.lmplot(data=results_2_plot, x="Turbidity (NTU)", y="Conductivity (µS/cm)")
plt.title("Predicted Conductivity vs Turbidity (Two-Feature Model)")
plt.tight_layout()
plt.show()

### Multi-Parameter Model (All Features)

In [ ]:
features_all = ["pH", "Temperature (°C)", "Turbidity (NTU)", "Dissolved Oxygen (mg/L)"]
X_all = df[features_all]

model_full = linear_model.LinearRegression()
model_full.fit(X_all, y)
predictions_full = model_full.predict(X_all)

print(f"Intercept: {model_full.intercept_:.4f}")
print("Coefficients:")
for feat, coef in zip(features_all, model_full.coef_):
    print(f"  {feat}: {coef:.4f}")

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

r2 = r2_score(y, predictions_full)
rmse = np.sqrt(mean_squared_error(y, predictions_full))

print(f"R-squared: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

df["Predicted Conductivity (µS/cm)"] = predictions_full
df[["Sample ID", "Conductivity (µS/cm)", "Predicted Conductivity (µS/cm)"]].head(10)

## 8. Statistical Modeling (OLS)

Use ordinary least squares (OLS) regression from statsmodels for detailed statistical inference, including p-values, confidence intervals, and R-squared.

In [ ]:
X_ph = sm.add_constant(df["pH"].values)
y_temp = df["Temperature (°C)"].values

ols_model_1 = sm.OLS(y_temp, X_ph).fit()
print(ols_model_1.summary())

In [ ]:
X_turbidity = sm.add_constant(df["Turbidity (NTU)"].values)
y_conductivity = df["Conductivity (µS/cm)"].values

ols_model_2 = sm.OLS(y_conductivity, X_turbidity).fit()
print(ols_model_2.summary())

## 9. Conclusions

**Key findings from this analysis:**

1. **Strongest correlation**: pH and dissolved oxygen show a strong positive correlation (r = 0.705), suggesting that higher pH levels are associated with higher dissolved oxygen concentrations.

2. **Distribution characteristics**: All parameters follow approximately normal distributions within acceptable water quality ranges.

3. **Predictive modeling**: A multi-parameter linear regression model using pH, temperature, turbidity, and dissolved oxygen can predict conductivity values.

4. **Statistical significance**: OLS regression confirms statistically significant relationships between several parameter pairs (p < 0.05).

5. **Parameter relationships**: Temperature and conductivity show a moderate correlation, while turbidity has varying relationships with other parameters.